# 01 — Dataset Exploration

Characterize the 11 datasets used in this project: degree distributions, homophily, class balance.

**Homophilic:** Cora, Citeseer, Pubmed, Flickr, Arxiv, Yelp, Reddit, BBBP, Proteins.  
**Heterophilic:** Squirrel, Wisconsin.  
**Custom (deferred):** Ethereum AML graph — added in a later iteration.

Output of this notebook motivates the dataset choices in `proposal.tex`.

## 1. Datasets used in this project

Eleven benchmarks, drawn from the papers cited in the proposal. Ethereum is excluded for now.

| # | Dataset    | Task                              | Type         | Cited by               |
|---|------------|-----------------------------------|--------------|------------------------|
| 1 | Cora       | node classification               | homophilic   | Khedri et al. 2025     |
| 2 | Citeseer   | node classification               | homophilic   | (standard companion)   |
| 3 | Pubmed     | node classification               | homophilic   | (standard companion)   |
| 4 | Flickr     | node classification               | homophilic   | Zhou et al. 2021       |
| 5 | ogbn-arxiv | node classification               | homophilic   | Zhou et al. 2021       |
| 6 | Yelp       | node classification (multi-label) | homophilic   | Zhou et al. 2021       |
| 7 | Reddit     | node classification               | homophilic   | Zhou et al. 2021       |
| 8 | BBBP       | graph classification (molecules)  | homophilic   | Khedri et al. 2025     |
| 9 | Proteins   | graph classification              | homophilic   | Khedri et al. 2025     |
|10 | Squirrel   | node classification               | heterophilic | Liu et al. 2024 (CGP)  |
|11 | Wisconsin  | node classification               | heterophilic | Liu et al. 2024 (CGP)  |

Note: BBBP and Proteins are graph-classification benchmarks (collections of small graphs), so degree/homophily statistics are aggregated per-graph rather than over a single graph.

## 2. Where each dataset comes from

All datasets are public and download on first use; nothing is bundled in the repo.

- **PyTorch Geometric** (`torch_geometric.datasets`, 10 of 11):
  - `Planetoid` — Cora, Citeseer, Pubmed
  - `Flickr` — GraphSAINT Flickr split
  - `Yelp` — GraphSAINT Yelp split (multi-label)
  - `Reddit2` — sparse-feature Reddit (GraphSAINT)
  - `MoleculeNet(name='BBBP')` — blood-brain-barrier penetration; requires `rdkit` for SMILES parsing
  - `TUDataset(name='PROTEINS')` — Dobson & Doig protein graphs
  - `WikipediaNetwork(name='squirrel', geom_gcn_preprocess=True)` — heterophilic Squirrel split from Pei et al. (ICLR 2020)
  - `WebKB(name='Wisconsin')` — heterophilic Wisconsin webpage graph
- **Open Graph Benchmark** (`ogb.nodeproppred`, 1 of 11):
  - `PygNodePropPredDataset(name='ogbn-arxiv')` — arXiv citation network

Each dataset is downloaded once and cached under `data/raw/<family>/`, which is gitignored. Yelp, Reddit, and ogbn-arxiv are the heavy ones (hundreds of MB).

## 3. How loading is wired into the project

All loader code lives in [`src/gnn_pruning/data/loaders.py`](../../src/gnn_pruning/data/loaders.py). Each dataset has a thin `load_<name>(root)` wrapper around its PyG / OGB constructor; the wrappers are registered in `DATASET_REGISTRY`, and a single `load_dataset(name, root='data/raw')` dispatch picks the right one by canonical lowercase name (e.g. `"cora"`, `"ogbn-arxiv"`).

Static metadata (task type, homophily class, citation) lives next to the loaders in `DATASET_META`, so notebooks, the CLI, and tests share one source of truth.

Typical usage from anywhere in the project:

```python
from gnn_pruning.data import load_dataset
data = load_dataset("cora")
```

## 4. One-time install

Run once in your venv from the repo root (the dependency list lives in `pyproject.toml`):

```sh
pip install -e ".[dev]"
```

This pulls in `torch`, `torch_geometric`, `ogb`, `rdkit`, and the notebook tooling.

In [ ]:
# Resolve the repo root so the notebook works regardless of where Jupyter was launched.
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Could not locate repo root (no pyproject.toml found upward).")
    REPO_ROOT = REPO_ROOT.parent

DATA_ROOT = REPO_ROOT / "data" / "raw"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {REPO_ROOT}")
print(f"Cache dir: {DATA_ROOT}")

In [ ]:
# Load all 11 datasets. First run downloads everything (Yelp / Reddit / ogbn-arxiv are large);
# subsequent runs hit the on-disk cache and return immediately.
from gnn_pruning.data import DATASET_REGISTRY, load_dataset

datasets = {name: load_dataset(name, root=DATA_ROOT) for name in DATASET_REGISTRY}
list(datasets)

In [ ]:
# Sanity table: shape + task metadata for each dataset.
import pandas as pd
from torch_geometric.data import Data, InMemoryDataset

from gnn_pruning.data import DATASET_META


def summarize(name, obj):
    meta = DATASET_META[name]
    row = {
        "dataset": meta.name,
        "task": meta.task,
        "homophily": meta.homophily,
        "source": meta.source,
    }
    if isinstance(obj, Data):
        row["n_graphs"] = 1
        row["n_nodes"] = int(obj.num_nodes)
        row["n_edges"] = int(obj.num_edges)
        row["n_features"] = int(obj.num_node_features)
        # ogbn-arxiv stores labels as (N, 1); flatten before counting.
        if obj.y is not None:
            y = obj.y.view(-1) if obj.y.dim() > 1 else obj.y
            row["n_classes"] = int(y.max().item()) + 1 if y.dtype.is_floating_point is False else y.size(-1)
        else:
            row["n_classes"] = None
    else:  # InMemoryDataset (BBBP, Proteins)
        row["n_graphs"] = len(obj)
        row["n_nodes"] = sum(g.num_nodes for g in obj)
        row["n_edges"] = sum(g.num_edges for g in obj)
        row["n_features"] = obj.num_node_features
        row["n_classes"] = obj.num_classes
    return row


summary = pd.DataFrame([summarize(name, obj) for name, obj in datasets.items()])
summary

## 5. Next steps

Subsequent cells in this notebook will use `datasets` to compute:

- Degree distributions (mean, median, max, tail behavior) — the topology signal that drives activation magnitudes in the message-passing recurrence.
- Homophily ratios — to confirm the homophilic / heterophilic split above and quantify the gap.
- Class balance — to flag where macro-F1 (or per-class metrics) will be needed instead of accuracy.

These statistics feed directly into the Wanda-Per-Class and Wanda-Degree-Weighted scoring variants defined in `proposal.tex`.